<a href="https://colab.research.google.com/github/gaurizendekar/Data_Science_labs/blob/main/Exp7_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Experiment 7: Sequence-to-Sequence Model with Real Text Data

Aim: To implement a Sequence-to-Sequence model using LSTM with real text sequence data in TensorFlow and Keras.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Sample Text Dataset
input_texts = [
    "hello",
    "how are you",
    "good morning",
    "thank you",
    "see you"
]

# Adding <start> and <end> tokens for proper Seq2Seq training (Teacher Forcing)
target_texts_inputs = [
    "<start> hi",
    "<start> i am fine",
    "<start> morning",
    "<start> welcome",
    "<start> bye"
]

target_texts_outputs = [
    "hi <end>",
    "i am fine <end>",
    "morning <end>",
    "welcome <end>",
    "bye <end>"
]

# 2. Tokenizer (combining all texts to build the vocabulary)
# Setting filters='' ensures <start> and <end> aren't stripped of their brackets
tokenizer = Tokenizer(filters='') 
tokenizer.fit_on_texts(input_texts + target_texts_inputs + target_texts_outputs)

# Convert Text to Sequences
input_sequences = tokenizer.texts_to_sequences(input_texts)
decoder_input_sequences = tokenizer.texts_to_sequences(target_texts_inputs)
decoder_target_sequences = tokenizer.texts_to_sequences(target_texts_outputs)

# 3. Padding
max_len_input = max(len(seq) for seq in input_sequences)
max_len_target = max(len(seq) for seq in decoder_input_sequences)

encoder_input_data = pad_sequences(input_sequences, maxlen=max_len_input, padding='post')
decoder_input_data = pad_sequences(decoder_input_sequences, maxlen=max_len_target, padding='post')
decoder_target_data = pad_sequences(decoder_target_sequences, maxlen=max_len_target, padding='post')

# Vocabulary Size (+1 for the 0 padding token)
vocab_size = len(tokenizer.word_index) + 1

# One-Hot Encode Target for the Categorical Crossentropy Loss
decoder_target_one_hot = tf.keras.utils.to_categorical(decoder_target_data, num_classes=vocab_size)

# 4. Build Seq2Seq Architecture
latent_dim = 64

# --- Encoder ---
encoder_inputs = Input(shape=(max_len_input,))
encoder_embedding = Embedding(input_dim=vocab_size, output_dim=latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# We discard encoder_outputs and only keep the states
encoder_states = [state_h, state_c] 

# --- Decoder ---
decoder_inputs = Input(shape=(max_len_target,))
decoder_embedding = Embedding(input_dim=vocab_size, output_dim=latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

# Decoder uses encoder_states as its initial state
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# --- Define the Model ---
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

# 5. Compile Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 6. Train Model
# verbose=0 suppresses spam output for 200 epochs, relying on graphs instead
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_one_hot,
    batch_size=2,
    epochs=150,
    verbose=0 
)

# 7. Evaluate Model
loss, accuracy = model.evaluate([encoder_input_data, decoder_input_data], decoder_target_one_hot, verbose=0)
print(f"\nModel Loss : {loss:.4f}")
print(f"Model Accuracy : {accuracy:.4f}")

# 8. Plot Accuracy and Loss Graphs (Side-by-Side View)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plot Accuracy
ax[0].plot(history.history['accuracy'], label='Training Accuracy', color='blue')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].set_title('Model Accuracy')
ax[0].legend()
ax[0].grid(True)

# Plot Loss
ax[1].plot(history.history['loss'], label='Training Loss', color='blue')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].set_title('Model Loss')
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 3, 64)     │        960 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 3, 64)     │        960 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 64),      │     33,024 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 3, 64),   │     33,024 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm_2[0][1],     │
│                     │ (None, 64)]       │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3, 15)     │        975 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 68,943 (269.31 KB)

 Trainable params: 68,943 (269.31 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.0667 - loss: 2.7038  
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 2.6735 
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.6428
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5333 - loss: 2.6111
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.5667 
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5333 - loss: 2.5133
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.4445
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 2.3651
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.2330
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.0651
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 1.8528
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5333 

Conclusion: Successfully implemented a Sequence-to-Sequence model using real text sequence data with Encoder-Decoder LSTM architecture in TensorFlow and Keras.